In [3]:
!pip install -q langchain langchain-cohere cohere faiss-cpu gradio


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.1/45.1 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.3/334.3 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 57.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 63.5 MB/s eta 0:00:00


In [4]:
from google.colab import drive, userdata
import pandas as pd
import os
import numpy as np
import faiss
import cohere
import gradio as gr
from langchain_cohere import ChatCohere
from langchain_core.messages import HumanMessage, SystemMessage

In [5]:

#CELDA 3 — Cargar CSV (correr UNA sola vez)
drive.mount('/content/drive')
ruta_archivo = "/content/drive/MyDrive/Alura_challenge/archivo.csv"
df = pd.read_csv(ruta_archivo, encoding='utf-8-sig')
print(f"✅ Archivo cargado. Filas: {len(df)}, Columnas: {list(df.columns)}")



Mounted at /content/drive
✅ Archivo cargado. Filas: 199, Columnas: ['SKU', 'Cód. de Barras (EAN)', 'Descripción', 'Marca', 'Categoría', 'Subcategoría', 'UN', 'Ubicación', 'Stock Actual', 'Stock Mínimo', 'Stock Máximo', 'Lote', 'Fecha de Fabricación', 'Fecha de Vencimiento', 'Costo Unitario', 'Precio de Venta Unitario', 'Proveedor Principal', 'Tiempo de Reposición']


In [6]:

# CELDA 4 — Configurar Cohere (correr UNA sola vez)
os.environ["COHERE_API_KEY"] = userdata.get('COHERE_API_KEY')
co  = cohere.Client(os.environ["COHERE_API_KEY"])           # para embeddings
llm = ChatCohere(model="command-r-08-2024", temperature=0)  # para respuestas
print("✅ Cohere configurado")



✅ Cohere configurado


In [7]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELDA 5 — Embeddings inteligentes: carga o regenera según necesidad   ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import os

RUTA_BASE      = "/content/drive/MyDrive/Alura_challenge/"
RUTA_EMBEDDINGS = RUTA_BASE + "embeddings.npy"
RUTA_INDEX      = RUTA_BASE + "index.faiss"
RUTA_CSV        = ruta_archivo  # ya definida en celda 3

def embeddings_desactualizados() -> bool:
    """Devuelve True si el CSV es más nuevo que los embeddings guardados."""
    if not os.path.exists(RUTA_EMBEDDINGS) or not os.path.exists(RUTA_INDEX):
        return True  # no existen → hay que generarlos
    fecha_csv        = os.path.getmtime(RUTA_CSV)
    fecha_embeddings = os.path.getmtime(RUTA_EMBEDDINGS)
    return fecha_csv > fecha_embeddings  # CSV más nuevo → regenerar

if embeddings_desactualizados():
    print("⚙️  Generando embeddings (CSV nuevo o primera vez)...")

    df["texto_busqueda"] = (
        df["Descripción"].fillna("") + " " +
        df["Marca"].fillna("") + " " +
        df["Categoría"].fillna("") + " " +
        df["Subcategoría"].fillna("")
    ).str.strip()

    response = co.embed(
        texts=df["texto_busqueda"].tolist(),
        model="embed-multilingual-v3.0",
        input_type="search_document"
    )
    embeddings = np.array(response.embeddings).astype("float32")

    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(embeddings)

    # Guardar para la próxima sesión
    np.save(RUTA_EMBEDDINGS, embeddings)
    faiss.write_index(index, RUTA_INDEX)
    print(f"✅ Embeddings generados y guardados. {index.ntotal} productos indexados.")

else:
    print("📂 Cargando embeddings existentes desde Drive...")
    embeddings = np.load(RUTA_EMBEDDINGS)
    index = faiss.read_index(RUTA_INDEX)
    print(f"✅ Cargados. {index.ntotal} productos indexados.")

📂 Cargando embeddings existentes desde Drive...
✅ Cargados. 199 productos indexados.


In [8]:
# CELDA 6 — Funciones del agente (correr UNA sola vez)
def buscar_productos(pregunta: str, top_k: int = 5) -> pd.DataFrame:
    """Embeddea solo la pregunta y busca las filas más similares en FAISS."""
    emb_pregunta = co.embed(
        texts=[pregunta],
        model="embed-multilingual-v3.0",
        input_type="search_query"
    ).embeddings
    emb_pregunta = np.array(emb_pregunta).astype("float32")
    _, indices = index.search(emb_pregunta, top_k)
    columnas = [c for c in df.columns if c != "texto_busqueda"]  # ← definida acá adentro
    return df.iloc[indices[0]][columnas]

def preguntar(pregunta: str) -> str:
    """Busca contexto relevante y consulta al LLM con solo esas filas."""
    filas = buscar_productos(pregunta, top_k=5)
    contexto = filas.to_string(index=False)

    mensajes = [
        SystemMessage(content=(
            "Eres un asistente que analiza datos de un catálogo de productos. "
            "Respondé en español, de forma clara y concisa. "
            "Solo usá la información del contexto proporcionado. "
            "Si la respuesta no está en el contexto, decilo explícitamente."
        )),
        HumanMessage(content=f"Contexto (productos relevantes):\n{contexto}\n\nPregunta: {pregunta}")
    ]
    return llm.invoke(mensajes).content

print("✅ Funciones listas. Podés empezar a preguntar.")

✅ Funciones listas. Podés empezar a preguntar.


In [9]:
import gradio as gr
import json
import time
from datetime import datetime

LOG_PATH = "/content/drive/MyDrive/Alura_challenge/logs.json"

def guardar_log(pregunta, productos, respuesta, tiempo):
    entrada = {
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "pregunta": pregunta,
        "productos_recuperados": productos,
        "respuesta": respuesta,
        "tiempo_respuesta_seg": round(tiempo, 2)
    }
    logs = []
    try:
        with open(LOG_PATH, "r") as f:
            logs = json.load(f)
    except:
        pass
    logs.append(entrada)
    with open(LOG_PATH, "w") as f:
        json.dump(logs, f, ensure_ascii=False, indent=2)

def preguntar_con_historial(pregunta: str, historial: list) -> tuple:
    inicio = time.time()

    filas = buscar_productos(pregunta, top_k=5)
    contexto = filas.to_string(index=False)
    productos_recuperados = filas["Descripción"].tolist()

    mensajes = [
        SystemMessage(content=(
            "Eres un asistente que analiza datos de un catálogo de productos. "
            "Respondé en español, de forma clara y concisa. "
            "Solo usá la información del contexto proporcionado. "
            "Si la respuesta no está en el contexto, decilo explícitamente. "
            "Tenés memoria de la conversación actual."
        ))
    ]

    for msg in historial:
        if msg["role"] == "user":
            mensajes.append(HumanMessage(content=msg["content"]))
        else:
            mensajes.append(SystemMessage(content=msg["content"]))

    mensajes.append(
        HumanMessage(content=f"Contexto (productos relevantes):\n{contexto}\n\nPregunta: {pregunta}")
    )

    respuesta = llm.invoke(mensajes).content
    tiempo = time.time() - inicio

    guardar_log(pregunta, productos_recuperados, respuesta, tiempo)

    historial.append({"role": "user", "content": pregunta})
    historial.append({"role": "assistant", "content": respuesta})
    return "", historial, historial.copy()

In [ ]:
with gr.Blocks(title="Agente de Productos") as demo:
    gr.Markdown("## 🛒 Agente de Productos\nHacé preguntas sobre el catálogo en lenguaje natural.")

    chatbot = gr.Chatbot(label="Conversación", height=400)
    historial_state = gr.State([])

    with gr.Row():
        txt_input = gr.Textbox(
            placeholder="Ej: ¿Qué precio tiene el arroz integral?",
            label="Tu pregunta",
            scale=4
        )
        btn_enviar = gr.Button("Enviar", scale=1)

    btn_limpiar = gr.Button("🗑️ Limpiar conversación")

    txt_input.submit(
        preguntar_con_historial,
        inputs=[txt_input, historial_state],
        outputs=[txt_input, chatbot, historial_state]
    )
    btn_enviar.click(
        preguntar_con_historial,
        inputs=[txt_input, historial_state],
        outputs=[txt_input, chatbot, historial_state]
    )

    btn_limpiar.click(lambda: ([], []), outputs=[chatbot, historial_state])

demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://1b48dbdaa8a09e2324.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/usr/local/lib/python3.12/dist-packages/gradio/routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


In [ ]:
import gradio as gr
print(gr.__version__)

6.19.0
